In [8]:
# Mount Drive e setup cartelle
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/Progetto_EVWSD_ML"
EMBEDDINGS_DIR = os.path.join(DRIVE_PROJECT_PATH, "embeddings")
HF_CACHE_DIR = os.path.join(DRIVE_PROJECT_PATH, "data/hf_cache")

os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)

print("Cartelle pronte su Drive:", EMBEDDINGS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cartelle pronte su Drive: /content/drive/MyDrive/Progetto_EVWSD_ML/embeddings


In [9]:
!pip install -q sentence-transformers datasets pillow torch

In [10]:
#  Caricamento dataset da HuggingFace
from datasets import load_dataset

# dataset di training rilasciato per EVWSD-ITA
dataset = load_dataset(
    "swap-uniba/EVWSD-ITA",
    cache_dir=HF_CACHE_DIR
)

print(dataset)
# ispeziona la prima istanza per capire la struttura
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'hyp_id', 'gloss', 'lemma', 'hyp_lemma', 'bns', 'is_co_hyp', 'images', 'all_lemmas', 'all_glosses', 'img'],
        num_rows: 10000
    })
})
{'id': 'bn:00022412n', 'hyp_id': 'bn:00017670n', 'gloss': 'Atto del cuocere', 'lemma': 'cucina', 'hyp_lemma': ['cambiamento di stato'], 'bns': ['bn:00018237n', 'bn:00014468n', 'bn:00037474n', 'bn:00022423n', 'bn:00024323n', 'bn:00049248n'], 'is_co_hyp': [True, False, False, False, False, False], 'images': ['F14/bn:00018237n', 'F0/bn:00014468n', 'F0/bn:00037474n', 'F24/bn:00022423n', 'F26/bn:00024323n', 'F0/bn:00049248n'], 'all_lemmas': [['masticazione', 'masticare'], ['nave cambusa', 'cucina', 'cambusa'], ['culinaria', 'cucina', 'gastronomiche', 'gastronomo', 'gastronomia', 'gastronomica', 'arte culinaria', 'gastronomico'], ['cottura', 'cucina', 'stufa a legna', 'cucina a gas', 'cucina fornello', 'stufa cuoco', 'fornello', 'piano cottura'], ['culinaria', 'cottura', 'cucina', 'cucinare', 

In [13]:
from huggingface_hub import HfApi, list_repo_files

api = HfApi()
file_list = list_repo_files("swap-uniba/EVWSD-ITA", repo_type="dataset")

print("Numero file totali:", len(file_list))
print("\nPrimi 30 file:")
for f in file_list[:30]:
    print(f)

print("\nEstensioni trovate:")
exts = set(f.split(".")[-1] for f in file_list if "." in f.split("/")[-1])
print(exts)

Numero file totali: 4

Primi 30 file:
.gitattributes
README.md
ds_train.json
imgs.zip

Estensioni trovate:
{'md', 'zip', 'gitattributes', 'json'}


In [14]:
# cerca specificamente file che contengano il path che conosciamo
target = "F22/bn:00022412n"
matches = [f for f in file_list if target in f]
print("Match trovati per", target, ":", matches)

Match trovati per F22/bn:00022412n : []


In [ ]:
# Caricamento modello CLIP multilingue (baseline)
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = SentenceTransformer(
    "sentence-transformers/clip-ViT-B-32-multilingual-v1",
    device=device,
    cache_folder=HF_CACHE_DIR
)

In [ ]:
# Funzione di estrazione embeddings immagini (con checkpoint su Drive)
import torch
from PIL import Image
from tqdm import tqdm

def compute_image_embeddings(dataset_split, model, batch_size=32, checkpoint_path=None):
    """
    Calcola gli embedding CLIP per tutte le immagini candidate di ogni istanza.
    Salva un checkpoint su Drive ogni tot batch per evitare di perdere lavoro
    se la sessione Colab si disconnette.
    """
    all_embeddings = []

    # adatta 'images' al nome reale del campo una volta ispezionata la struttura (Cella 3)
    images_field = dataset_split["images"] if "images" in dataset_split.column_names else None

    if images_field is None:
        raise ValueError("Campo immagini non trovato: controlla i nomi delle colonne nel dataset")

    for start in tqdm(range(0, len(images_field), batch_size)):
        batch_images = images_field[start:start+batch_size]
        with torch.no_grad():
            batch_emb = model.encode(
                batch_images,
                batch_size=batch_size,
                convert_to_tensor=True,
                device=model.device,
                show_progress_bar=False
            )
        all_embeddings.append(batch_emb.cpu())

        if checkpoint_path and (start // batch_size) % 20 == 0:
            torch.save(torch.cat(all_embeddings, dim=0), checkpoint_path)

    final_tensor = torch.cat(all_embeddings, dim=0)
    if checkpoint_path:
        torch.save(final_tensor, checkpoint_path)
    return final_tensor

In [ ]:
# Esecuzione e salvataggio finale
tensor_path = os.path.join(EMBEDDINGS_DIR, "tensor_immagini.pt")

embeddings_immagini = compute_image_embeddings(
    dataset["train"],
    model,
    batch_size=32,
    checkpoint_path=tensor_path
)

print("Shape finale:", embeddings_immagini.shape)
print("Salvato in:", tensor_path)